In [2]:
from pprint import pprint
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.redis import RedisSaver
from langgraph.store.redis import RedisStore

from agentic_patterns.react_agent.agent import REACT_AGENT_BUILDER
redis_url = "redis://localhost:6379/0"
thread_id = "019db05b-da01-72e0-bf19-b0ebe0e3d3a3"
with RedisStore.from_conn_string(redis_url) as store:
    with RedisSaver.from_conn_string(redis_url) as ch:
        config = RunnableConfig(configurable={"thread_id": thread_id})        
        agent = REACT_AGENT_BUILDER.compile(checkpointer=ch, store=store)

        history = [*agent.get_state_history(config=config)]
        # pprint(history, indent=2)
        if not history:
            print(f"No history found for thread_id {thread_id}.")
        else:
            last_state = history[0]
            print(len(last_state.values.get("messages") or []))
            pprint(last_state.interrupts, indent=2)
        # ch.delete_thread(thread_id)
         

ModuleNotFoundError: No module named 'agentic_patterns.react_agent'

In [4]:
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.redis import AsyncRedisSaver

from agentic_patterns.subagent_pattern.checkpointer_modes_experiment import (
    REDIS_URL,
    install_safe_pending_sends_loader,
)

install_safe_pending_sends_loader()

MODE = "own_same_thread"
parent_thread_id = f"ckpt-mode-{MODE}"
parent_config = RunnableConfig(configurable={"thread_id": parent_thread_id})

async with AsyncRedisSaver.from_conn_string(REDIS_URL) as ch:
    await ch.asetup()
    chp_list = [t async for t in ch.alist(parent_config)]

print(f"alist returned {len(chp_list)} CheckpointTuple(s) for parent_config={parent_config}\n")
pprint(chp_list[-6], indent=2)
# Per-tuple summary, oldest -> newest
for i, tup in enumerate(reversed(chp_list)):
    cfg = tup.config["configurable"]
    msgs = tup.checkpoint.get("channel_values", {}).get("messages", [])
    print(
        f"[{i}] thread={cfg['thread_id']}  "
        f"ns={cfg.get('checkpoint_ns', '')!r}  "
        f"ckpt_id={cfg['checkpoint_id'][-15:]}  "
        f"parent_ckpt_id={tup.metadata.get('parents', {}).get("", [])[-15:]}  "
        f"step={tup.metadata.get('step'):<3} "
        f"source={tup.metadata.get('source'):<8} "
        f"msgs={len(msgs)}"
    )

# Group by location so it's easy to see which checkpoints belong to parent vs subagent
print("\n--- distinct (thread_id, checkpoint_ns) seen ---")
by_loc: dict[tuple[str, str], int] = {}
for tup in chp_list:
    key = (
            tup.config["configurable"]["thread_id"],
            tup.config["configurable"].get("checkpoint_ns", ""),
            tup.config["configurable"].get("checkpoint_id", ""),
        )
    by_loc[key] = by_loc.get(key, 0) + 1
for (t, n, ch_id), c in sorted(by_loc.items()):
    print(f"  thread={t}  ns={n!r}  ckpt_id={ch_id[-10:]}  count={c}")

# Peek at one full tuple structure
if chp_list:
    head = chp_list[0]
    print("\n--- head CheckpointTuple structure ---")
    print(f"  config.configurable    : {head.config['configurable']}")
    print(f"  checkpoint top-level   : {list(head.checkpoint.keys())}")
    print(f"  metadata               : {dict(head.metadata or {})}")
    print(f"  channel_values keys    : {list(head.checkpoint.get('channel_values', {}).keys())}")
    print(f"  parent_config          : {head.parent_config}")
    print(f"  pending_writes count   : {len(head.pending_writes or [])}")
    if head.pending_writes:
        print(f"  pending_writes[0]      : {head.pending_writes[0]}")


alist returned 15 CheckpointTuple(s) for parent_config={'configurable': {'thread_id': 'ckpt-mode-own_same_thread'}}

CheckpointTuple(config={'configurable': {'thread_id': 'ckpt-mode-own_same_thread', 'checkpoint_ns': 'tools:7857bbf2-264a-9928-667b-6e8b2f84d7de', 'checkpoint_id': '1f1483ee-e3bb-64ec-bfff-89047ac02233'}}, checkpoint={'type': 'json', 'v': 4, 'ts': '2026-05-05T04:57:34.725847+00:00', 'id': '1f1483ee-e3bb-64ec-bfff-89047ac02233', 'channel_values': {'__start__': {'messages': [{'role': 'user', 'content': 'Tell me one fact about carrots.'}]}}, 'channel_versions': {'__start__': '00000000000000000000000000000001.0.7794106066210351'}, 'versions_seen': {'__input__': {}}, 'updated_channels': ['__start__'], 'pending_sends': []}, metadata={'source': 'input', 'step': -1, 'parents': {'': '1f1483ee-e38a-6c78-8001-6fbd457880a4'}}, parent_config=None, pending_writes=[('5da665bb-9b92-a782-e6ce-8b1cec4db789', 'messages', [{'role': 'user', 'content': 'Tell me one fact about carrots.'}]), ('5

In [8]:
import base64
from typing import Any

import orjson
from redis.asyncio import Redis

from agentic_patterns.subagent_pattern.checkpointer_modes_experiment import REDIS_URL

MODE = "own_same_thread"
thread_id = f"ckpt-mode-{MODE}"


def try_decode_blob(b64: str) -> tuple[bool, str | bytes | None]:
    try:
        raw = base64.b64decode(b64)
    except Exception as e:
        return False, f"base64: {e}"
    try:
        orjson.loads(raw)
    except Exception:
        return False, raw
    return True, raw


def walk_doc(node: Any, path: str, out: list[dict]) -> None:
    if isinstance(node, dict):
        if "blob" in node and isinstance(node["blob"], str):
            ok, payload = try_decode_blob(node["blob"])
            if not ok:
                out.append({"path": f"{path}.blob", "type": node.get("type"),
                            "channel": node.get("channel"), "err_or_raw": payload})
        if "__bytes__" in node and isinstance(node["__bytes__"], str):
            ok, payload = try_decode_blob(node["__bytes__"])
            if not ok:
                out.append({"path": f"{path}.__bytes__", "type": "bytes-marker",
                            "channel": None, "err_or_raw": payload})
        for k, v in node.items():
            walk_doc(v, f"{path}.{k}", out)
    elif isinstance(node, list):
        for i, v in enumerate(node):
            walk_doc(v, f"{path}[{i}]", out)


PREFIXES = ["checkpoint", "checkpoint_write", "checkpoint_latest"]
totals: dict[str, dict[str, int]] = {p: {"docs": 0, "json": 0, "bad": 0} for p in PREFIXES}
all_bad: list[dict] = []
non_json: dict[str, list[str]] = {p: [] for p in PREFIXES}

async with Redis.from_url(REDIS_URL, decode_responses=False) as r:
    for prefix in PREFIXES:
        keys = [k async for k in r.scan_iter(match=f"{prefix}:{thread_id}*".encode())]
        totals[prefix]["docs"] = len(keys)
        for k in keys:
            ktype = (await r.type(k)).decode()
            if ktype != "ReJSON-RL":
                non_json[prefix].append(f"{k.decode()}  (type={ktype})")
                continue
            totals[prefix]["json"] += 1
            doc = await r.json().get(k)
            if not isinstance(doc, (dict, list)):
                continue
            findings: list[dict] = []
            walk_doc(doc, "", findings)
            if findings:
                totals[prefix]["bad"] += 1
                for f in findings:
                    f["key"] = k.decode()
                    f["prefix"] = prefix
                all_bad.extend(findings)

print(f"{'prefix':<22} {'docs':>6} {'json':>6} {'bad':>6}")
for p, t in totals.items():
    print(f"{p:<22} {t['docs']:>6} {t['json']:>6} {t['bad']:>6}")

for p, lst in non_json.items():
    if lst:
        print(f"\nnon-JSON keys under {p}:")
        for s in lst[:5]:
            print(f"  {s}")

print(f"\ntotal bad blobs: {len(all_bad)}\n")
for entry in all_bad[:6]:
    raw = entry["err_or_raw"]
    print(f"BAD {entry['prefix']}{entry['path']}  type={entry.get('type')}  channel={entry.get('channel')}")
    print(f"  key: {entry['key']}")
    if isinstance(raw, bytes):
        print(f"  len: {len(raw)}")
        print(f"  head: {raw[:200]!r}")
        print(f"  tail: {raw[-120:]!r}")
    else:
        print(f"  err: {raw}")
    print()


prefix                   docs   json    bad
checkpoint                 15     15      0
checkpoint_write           24     24      7
checkpoint_latest           3      0      0

non-JSON keys under checkpoint_latest:
  checkpoint_latest:ckpt-mode-own_same_thread:__empty__  (type=string)
  checkpoint_latest:ckpt-mode-own_same_thread:tools:08060305-8228-1835-65f0-274a7a10d87b  (type=string)
  checkpoint_latest:ckpt-mode-own_same_thread:tools:624b5240-a946-7c04-15d8-438d19b7a28b  (type=string)

total bad blobs: 7

BAD checkpoint_write.blob  type=null  channel=branch:to:model
  key: checkpoint_write:ckpt-mode-own_same_thread:tools:08060305-8228-1835-65f0-274a7a10d87b:1f147007-28c0-6c24-bfff-88e820726dbd:9183c2d6-f892-eaf2-4d99-86325da7f616:1
  len: 0
  head: b''
  tail: b''

BAD checkpoint_write.blob  type=null  channel=branch:to:model
  key: checkpoint_write:ckpt-mode-own_same_thread:__empty__:1f147007-07fa-6773-bfff-fa7db86a91de:f253aab9-4917-3829-93d6-f3106280c3bb:1
  len: 0
  head: b''
